In [131]:
import numpy as np

seed = 23
n = 20
rng = np.random.default_rng(seed)

In [132]:
def next_int(a, b):
    return int(rng.integers(a, b + 1))

def generate_instance(n):
    p = [next_int(1, 30) for _ in range(n)]
    w = [next_int(1, 30) for _ in range(n)]
    s = sum(p)
    d = [next_int(1, s) for _ in range(n)]

    return p, w, d

p, w, d = generate_instance(n)

In [133]:
class Subject:
    def __init__(self, permutation):
        self.permutation = permutation
        self.fitness = None

    def evaluate_fitness(self, p, w, d):
        time = 0
        total = 0
        for i in self.permutation:
            time += p[i]
            delay = max(0, time - d[i])
            total += w[i] * delay
        self.fitness = total

In [134]:
def init_population(size, n):
    population = []
    for _ in range(size):
        permutation = rng.permutation(n)
        subject = Subject(permutation)
        subject.evaluate_fitness(p, w, d)
        population.append(subject)
    return population

def tournament_selection(population, k=3):
    candidates = rng.choice(population, size=k, replace=False)
    return min(candidates, key=lambda x: x.fitness)

def ox_crossover(parent1, parent2):
    n = len(parent1.permutation)
    point1, point2 = sorted(rng.choice(n, size=2, replace=False))
    child = [-1] * n
    child[point1:point2] = parent1.permutation[point1:point2]
    remaining = [x for x in parent2.permutation if x not in child]
    positions = list(range(point2, n)) + list(range(0, point1))
    for pos, val in zip(positions, remaining):
        child[pos] = val

    return Subject(np.array(child))

def mutate(subject, mutation_rate=0.05):
    if rng.random() < mutation_rate:
        i, j = rng.choice(len(subject.permutation), size=2, replace=False)
        subject.permutation[i], subject.permutation[j] = subject.permutation[j], subject.permutation[i]
    return subject

In [135]:
_POPULATION_SIZE = 50
_GENERATIONS = 50
_CROSSOVER_PROBABILITY = 0.7
_MUTATION_PROBABILITY = 0.05

population = init_population(_POPULATION_SIZE, n)
best_history = []
average_history = []
worst_history = []

for generation in range(_GENERATIONS):
    new_population = []

    best_subject = min(population, key=lambda s: s.fitness)
    new_population.append(best_subject)

    while len(new_population) < _POPULATION_SIZE:
        parent1 = tournament_selection(population)
        parent2 = tournament_selection(population)

        if rng.random() < _CROSSOVER_PROBABILITY:
            child = ox_crossover(parent1, parent2)
        else:
            child = Subject(parent1.permutation.copy())

        child = mutate(child, _MUTATION_PROBABILITY)
        child.evaluate_fitness(p, w, d)
        new_population.append(child)

    population = new_population

    fitnesses = [s.fitness for s in population]
    best_history.append(min(fitnesses))
    average_history.append(sum(fitnesses) / len(fitnesses))
    worst_history.append(max(fitnesses))

    print(f"Generation {generation + 1}: best={min(fitnesses)}, avg={round(sum(fitnesses)/len(fitnesses))}, worst={max(fitnesses)}")


Generation 1: best=5740, avg=11264, worst=18907
Generation 2: best=5410, avg=10533, worst=15339
Generation 3: best=5376, avg=9793, worst=15565
Generation 4: best=5376, avg=10201, worst=17707
Generation 5: best=5253, avg=10757, worst=19409
Generation 6: best=5253, avg=10141, worst=19429
Generation 7: best=5253, avg=10151, worst=19841
Generation 8: best=5005, avg=9138, worst=17054
Generation 9: best=4248, avg=9057, worst=17702
Generation 10: best=4248, avg=8095, worst=15858
Generation 11: best=4034, avg=8796, worst=19317
Generation 12: best=4034, avg=9254, worst=18408
Generation 13: best=4034, avg=9935, worst=19440
Generation 14: best=3534, avg=9268, worst=17835
Generation 15: best=3514, avg=8641, worst=19352
Generation 16: best=3514, avg=9105, worst=19688
Generation 17: best=3514, avg=8340, worst=18779
Generation 18: best=3514, avg=8283, worst=19608
Generation 19: best=3514, avg=7946, worst=19131
Generation 20: best=2838, avg=7821, worst=18506
Generation 21: best=2838, avg=8849, worst=1